# PAM50 benchmark v1.1: control arms + per-seed gene tracking

Run top to bottom (Runtime > Run all). Inputs are the same `EXPORT_FOR_COLAB` files as v1.0.
Everything is written to a **new** folder `ML_RESULTS_v1.1` so nothing from v1.0 is overwritten.

Arms evaluated on identical splits per seed:
1. `pam50_centroid_v1` : the v1.0 PAM50 implementation, unchanged (continuity)
2. `pam50_genefu` : `genefu::intrinsic.cluster.predict(pam50.robust)` run in R on each test set alone
3. `lr_pam50genes` : logistic regression on the 50 PAM50 genes
4. `lr_full3000` : logistic regression on all 3,000 train-filtered genes
5. `compact_TOPk` : L1 selector + L2 LR, k = 20, 40, 50, 80, 120 (80 = primary)

In [ ]:
# Cell 1 — Setup + parameters
import os, json, time, subprocess, shutil, textwrap
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
except ImportError:                       # local test only
    DATA_DIR = os.environ.get("DATA_DIR", "./EXPORT_FOR_COLAB")

X_FP = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP = os.path.join(DATA_DIR, "y_labels.tsv")
OUT  = os.path.join(DATA_DIR, "ML_RESULTS_v1.1")
for sub in ["", "per_seed", "genefu", "stability", "biology", "env"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

SEEDS        = [123, 124, 125, 126, 127]
CLASS_ORDER  = ["Basal", "Her2", "LumA", "LumB"]
V            = 3000
C_SEL        = 0.6                      # selector strength, fixed a priori (as v1.0)
PANEL_SIZES  = [20, 40, 50, 80, 120]
PRIMARY_K    = 80
C_GRID       = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]   # same grid for every LR arm
STABLE_MIN   = 3                        # stable panel = selected in >= 3 of 5 seeds
N_BOOT       = 2000
RUN_GENEFU   = True
RUN_ENRICHR  = True

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))},
          open(os.path.join(OUT, "env", "run_config.json"), "w"), indent=2)
print("OUT =", OUT)

In [ ]:
# Cell 2 — Install R + genefu (once per Colab session, ~10-15 min) and export PAM50 centroids
if RUN_GENEFU:
    if shutil.which("Rscript") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "r-base", "r-base-dev"], check=True)
    r_install = '''
    options(repos = c(CRAN = "https://cloud.r-project.org"))
    if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
    if (!requireNamespace("genefu", quietly = TRUE)) BiocManager::install("genefu", update = FALSE, ask = FALSE)
    suppressPackageStartupMessages(library(genefu))
    suppressWarnings(try(data(pam50.robust, package = "genefu"), silent = TRUE))
    if (!exists("pam50.robust")) data(pam50, package = "genefu")
    stopifnot(exists("pam50.robust"))
    cent <- pam50.robust$centroids
    write.table(data.frame(GENE = toupper(rownames(cent)), cent, check.names = FALSE),
                file.path(Sys.getenv("OUT"), "genefu", "pam50_centroids.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
    cat("std =", pam50.robust$std, "| method.cor =", pam50.robust$method.cor, "| rescale.q =", pam50.robust$rescale.q, "\n")
    cat("genefu", as.character(packageVersion("genefu")), "\n")
    '''
    fp = os.path.join(OUT, "genefu", "install_and_export.R")
    open(fp, "w").write(textwrap.dedent(r_install))
    p = subprocess.run(["Rscript", fp], env={**os.environ, "OUT": OUT}, capture_output=True, text=True)
    open(os.path.join(OUT, "genefu", "install_and_export.log"), "w").write(p.stdout + p.stderr)
    print("\n".join((p.stdout + p.stderr).splitlines()[-6:]))
    assert p.returncode == 0, "R step failed, see genefu/install_and_export.log"

CENT_FP = os.path.join(OUT, "genefu", "pam50_centroids.tsv")
if not os.path.exists(CENT_FP):                       # fallback: v1.0 export
    CENT_FP = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED", "pam50_centroids.tsv")
cent = pd.read_csv(CENT_FP, sep="\t")
cent["GENE"] = cent["GENE"].str.upper().str.strip()
cent = cent.set_index("GENE")
print("Centroids:", cent.shape, "from", CENT_FP)

In [ ]:
# Cell 3 — Load data, log check, 4-class subset, PAM50 gene index
y_df = pd.read_csv(Y_FP, sep="\t")
X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
assert len(df) == len(y_df) and df["SAMPLE_ID"].is_unique

genes = np.array([c for c in df.columns if c not in ("SAMPLE_ID", "y")], dtype=str)
X_all = df[genes].to_numpy(np.float32)
y_all = df["y"].astype(str).to_numpy()
sid_all = df["SAMPLE_ID"].astype(str).to_numpy()

q50, q99 = np.nanpercentile(X_all, [50, 99])
if (np.nanmin(X_all) >= 0) and (np.nanmax(X_all) >= 500) and ((q99 + 1e-6) / (q50 + 1e-6) >= 20):
    X_all = np.log1p(X_all); print("log1p applied")
else:
    print("log1p skipped (values already log-scale)")

m4 = np.isin(y_all, CLASS_ORDER)
X4, sid4 = X_all[m4], sid_all[m4]
y4 = np.array([CLASS_ORDER.index(s) for s in y_all[m4]])
print("4-class:", X4.shape, dict(zip(CLASS_ORDER, np.bincount(y4))))

# PAM50 genes: centroid name -> column in X (aliases for renamed symbols)
ALIAS = {"CDCA1": "NUF2", "KNTC2": "NDC80", "ORC6L": "ORC6"}
g_up = {g.upper(): i for i, g in enumerate(genes)}
pam_map = {}
for g in cent.index:
    for cand in (g, ALIAS.get(g), {v: k for k, v in ALIAS.items()}.get(g)):
        if cand and cand in g_up:
            pam_map[g] = g_up[cand]; break
pam_names = np.array(list(pam_map))
pam_idx = np.array(list(pam_map.values()))
print(f"PAM50 genes matched: {len(pam_idx)}/{len(cent)}")
assert len(pam_idx) == 50, f"missing: {sorted(set(cent.index) - set(pam_names))}"
C_pam = cent.loc[pam_names, CLASS_ORDER].to_numpy(np.float32)

In [ ]:
# Cell 4 — Helpers
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             average_precision_score, recall_score, confusion_matrix)
from scipy.stats import binomtest
import warnings
warnings.filterwarnings("ignore", message="Unknown solver options")

def split_70_15_15(y, seed):                      # identical to v1.0 Cells 12/18
    idx = np.arange(len(y))
    tr, tmp = next(StratifiedShuffleSplit(1, test_size=0.30, random_state=seed).split(idx, y))
    va_r, te_r = next(StratifiedShuffleSplit(1, test_size=0.50, random_state=seed).split(tmp, y[tmp]))
    return tr, tmp[va_r], tmp[te_r]

def fit_prep(Xtr, *others):                       # train-only median impute + z-score
    imp, sc = SimpleImputer(strategy="median"), StandardScaler()
    out = [sc.fit_transform(imp.fit_transform(Xtr)).astype(np.float32)]
    return out + [sc.transform(imp.transform(X)).astype(np.float32) for X in others]

def lr(C, seed):
    return LogisticRegression(penalty="l2", solver="lbfgs", C=C, class_weight="balanced",
                              max_iter=5000, random_state=seed)

def tune_refit(Xtr, ytr, Xva, yva, Xte, seed):    # C on VAL (macro-F1), refit TRAIN+VAL, predict TEST
    scores = {C: f1_score(yva, lr(C, seed).fit(Xtr, ytr).predict(Xva), average="macro") for C in C_GRID}
    bestC = max(scores, key=scores.get)
    m = lr(bestC, seed).fit(np.vstack([Xtr, Xva]), np.concatenate([ytr, yva]))
    return m.predict_proba(Xte), bestC

def softmax(S):
    E = np.exp(S - S.max(1, keepdims=True)); return E / E.sum(1, keepdims=True)

def pam50_centroid_v1(Xtr_raw, Xte_raw):          # v1.0 rule: train median/MAD, Pearson, softmax
    A, B = Xtr_raw[:, pam_idx], Xte_raw[:, pam_idx]
    med = np.nanmedian(A, 0); mad = np.nanmedian(np.abs(A - med), 0) + 1e-6
    Z = np.nan_to_num((B - med) / mad)
    Zc = Z - Z.mean(1, keepdims=True); Cc = C_pam - C_pam.mean(0, keepdims=True)
    S = (Zc / (np.linalg.norm(Zc, axis=1, keepdims=True) + 1e-8)) @ (Cc / (np.linalg.norm(Cc, axis=0, keepdims=True) + 1e-8))
    return softmax(S)

def metrics(y, P):
    yhat = P.argmax(1)
    yb = label_binarize(y, classes=range(4))
    rec = recall_score(y, yhat, labels=range(4), average=None, zero_division=0)
    f1c = f1_score(y, yhat, labels=range(4), average=None, zero_division=0)
    out = {"macro_f1": f1_score(y, yhat, average="macro"),
           "bacc": balanced_accuracy_score(y, yhat),
           "macro_roc_auc": roc_auc_score(yb, P, average="macro"),
           "macro_pr_auc": average_precision_score(yb, P, average="macro")}
    out.update({f"recall_{c}": r for c, r in zip(CLASS_ORDER, rec)})
    out.update({f"f1_{c}": f for c, f in zip(CLASS_ORDER, f1c)})
    return out

In [ ]:
# Cell 5 — Splits for every seed (saved by SAMPLE_ID); check seed 123 matches v1.0
split_rows = []
for s in SEEDS:
    tr, va, te = split_70_15_15(y4, s)
    for part, ix in (("train", tr), ("val", va), ("test", te)):
        split_rows += [{"seed": s, "SAMPLE_ID": sid4[i], "part": part, "y": CLASS_ORDER[y4[i]]} for i in ix]
splits = pd.DataFrame(split_rows)
splits.to_csv(os.path.join(OUT, "splits_by_sample_id.tsv"), sep="\t", index=False)

old = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED", "primary4_split_seed123.npz")
if os.path.exists(old):
    o = np.load(old, allow_pickle=True)
    new_te = set(splits.query("seed == 123 and part == 'test'").SAMPLE_ID)
    print("Seed-123 test set identical to v1.0:", new_te == set(o["te_sid"].astype(str)))
print(splits.groupby(["seed", "part"]).size().unstack())

In [ ]:
# Cell 6 — Python arms on identical splits (+ panel genes per seed)
sid_pos = {s: i for i, s in enumerate(sid4)}
rows, preds, panels, tuned = [], [], [], []
t0 = time.time()

for s in SEEDS:
    sp = splits[splits.seed == s]
    tr, va, te = (np.array([sid_pos[x] for x in sp[sp.part == p].SAMPLE_ID]) for p in ("train", "val", "test"))
    Xtr_raw, Xva_raw, Xte_raw = X4[tr], X4[va], X4[te]
    ytr, yva, yte = y4[tr], y4[va], y4[te]
    P = {}

    # 1) v1.0 PAM50 centroid
    P["pam50_centroid_v1"] = pam50_centroid_v1(Xtr_raw, Xte_raw)

    # 3) LR on the 50 PAM50 genes
    a, b, c = fit_prep(Xtr_raw[:, pam_idx], Xva_raw[:, pam_idx], Xte_raw[:, pam_idx])
    P["lr_pam50genes"], C1 = tune_refit(a, ytr, b, yva, c, s)

    # train-only variance filter (V genes)
    v_idx = np.argsort(np.var(SimpleImputer(strategy="median").fit_transform(Xtr_raw), 0))[::-1][:V]
    Xtr, Xva, Xte = fit_prep(Xtr_raw[:, v_idx], Xva_raw[:, v_idx], Xte_raw[:, v_idx])

    # 4) LR on all V genes
    P["lr_full3000"], C2 = tune_refit(Xtr, ytr, Xva, yva, Xte, s)

    # 5) compact: L1 selector on TRAIN only, then L2 LR per panel size
    sel = LogisticRegression(penalty="l1", solver="saga", C=C_SEL, class_weight="balanced",
                             max_iter=1500, tol=1e-3, random_state=s).fit(Xtr, ytr)
    rank = np.argsort(np.abs(sel.coef_).mean(0))[::-1]
    Ck = {}
    for k in PANEL_SIZES:
        pi = rank[:k]
        P[f"compact_TOP{k}"], Ck[k] = tune_refit(Xtr[:, pi], ytr, Xva[:, pi], yva, Xte[:, pi], s)
    panels += [{"seed": s, "rank": r + 1, "GENE": g} for r, g in enumerate(genes[v_idx][rank[:PRIMARY_K]])]
    tuned.append({"seed": s, "C_lr_pam50genes": C1, "C_lr_full3000": C2,
                  "n_nonzero_selector_genes": int((np.abs(sel.coef_).sum(0) > 0).sum()),
                  **{f"C_compact_TOP{k}": c for k, c in Ck.items()}})

    for arm, Pm in P.items():
        rows.append({"seed": s, "arm": arm, **metrics(yte, Pm)})
        preds += [{"seed": s, "arm": arm, "SAMPLE_ID": sid4[i], "y_true": yte[j], "y_pred": int(Pm[j].argmax()),
                   **{f"p_{c}": Pm[j, q] for q, c in enumerate(CLASS_ORDER)}} for j, i in enumerate(te)]
    print(f"seed {s} done ({(time.time() - t0) / 60:.1f} min) | macro-F1 " +
          " ".join(f"{a}={m['macro_f1']:.3f}" for a, m in ((r['arm'], r) for r in rows if r['seed'] == s)))

per_seed = pd.DataFrame(rows)
pred_df = pd.DataFrame(preds)
pd.DataFrame(panels).to_csv(os.path.join(OUT, "stability", "compact_TOP80_genes_per_seed.tsv"), sep="\t", index=False)
pd.DataFrame(tuned).to_csv(os.path.join(OUT, "per_seed", "tuned_hyperparameters.tsv"), sep="\t", index=False)

In [ ]:
# Cell 7 — genefu arm: intrinsic.cluster.predict(pam50.robust) in R
#   (a) per seed, on each TEST set alone (no training data involved, labels never used)
#   (b) once on the full labelled cohort: concordance of genefu with the published METABRIC calls
if RUN_GENEFU:
    gdir = os.path.join(OUT, "genefu")
    Xg = pd.DataFrame(X_all[:, pam_idx], columns=pam_names)   # columns renamed to genefu symbols
    Xg.insert(0, "SAMPLE_ID", sid_all)
    Xg.to_csv(os.path.join(gdir, "pam50_expr_all_samples.tsv"), sep="\t", index=False)
    splits.query("part == 'test'")[["seed", "SAMPLE_ID"]].to_csv(os.path.join(gdir, "test_sets.tsv"), sep="\t", index=False)

    r_code = r'''
    suppressPackageStartupMessages(library(genefu))
    suppressWarnings(try(data(pam50.robust, package = "genefu"), silent = TRUE))
    if (!exists("pam50.robust")) data(pam50, package = "genefu")
    stopifnot(exists("pam50.robust"))
    d   <- Sys.getenv("GDIR")
    X   <- read.delim(file.path(d, "pam50_expr_all_samples.tsv"), check.names = FALSE)
    rownames(X) <- X$SAMPLE_ID; X$SAMPLE_ID <- NULL; X <- as.matrix(X)
    stopifnot(all(rownames(pam50.robust$centroids) %in% colnames(X)))
    annot <- data.frame(probe = colnames(X), EntrezGene.ID = NA, row.names = colnames(X))
    run <- function(M, tag) {
      r <- intrinsic.cluster.predict(sbt.model = pam50.robust, data = M, annot = annot,
                                     do.mapping = FALSE, do.prediction.strength = FALSE, verbose = FALSE)
      out <- data.frame(SAMPLE_ID = rownames(M), run = tag, call = as.character(r$subtype),
                        r$subtype.proba, check.names = FALSE)
      out
    }
    tests <- read.delim(file.path(d, "test_sets.tsv"))
    res <- run(X, "full_cohort")
    for (s in unique(tests$seed)) {
      ids <- as.character(tests$SAMPLE_ID[tests$seed == s])
      res <- rbind(res, run(X[ids, , drop = FALSE], paste0("seed_", s)))
    }
    write.table(res, file.path(d, "genefu_calls.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
    cat("genefu calls written:", nrow(res), "\n")
    '''
    fp = os.path.join(gdir, "run_genefu.R")
    open(fp, "w").write(textwrap.dedent(r_code))
    p = subprocess.run(["Rscript", fp], env={**os.environ, "GDIR": gdir}, capture_output=True, text=True)
    open(os.path.join(gdir, "run_genefu.log"), "w").write(p.stdout + p.stderr)
    print((p.stdout + p.stderr)[-1500:])
    assert p.returncode == 0, "genefu step failed, see genefu/run_genefu.log"

    gf = pd.read_csv(os.path.join(gdir, "genefu_calls.tsv"), sep="\t")

    def genefu_probs(sub):                        # 4-class: drop Normal column, renormalise
        Pm = sub[CLASS_ORDER].to_numpy(float)
        return Pm / Pm.sum(1, keepdims=True)

    # (a) per-seed arm
    for s in SEEDS:
        sub = gf[gf.run == f"seed_{s}"].set_index("SAMPLE_ID")
        te_ids = pred_df.query("seed == @s and arm == 'lr_full3000'").SAMPLE_ID.to_numpy()
        sub = sub.loc[te_ids]
        yte = np.array([CLASS_ORDER.index(v) for v in splits.query("seed == @s and part == 'test'").set_index("SAMPLE_ID").loc[te_ids, "y"]])
        Pm = genefu_probs(sub)
        per_seed = pd.concat([per_seed, pd.DataFrame([{"seed": s, "arm": "pam50_genefu", **metrics(yte, Pm),
                                                       "n_called_Normal": int((sub["call"] == "Normal").sum())}])])
        pred_df = pd.concat([pred_df, pd.DataFrame({"seed": s, "arm": "pam50_genefu", "SAMPLE_ID": te_ids, "y_true": yte,
                                                    "y_pred": Pm.argmax(1), **{f"p_{c}": Pm[:, q] for q, c in enumerate(CLASS_ORDER)}})])

    # (b) full-cohort concordance with published calls
    full = gf[gf.run == "full_cohort"].set_index("SAMPLE_ID").loc[sid_all]
    lab = pd.Series(y_all, index=sid_all)
    conc = pd.crosstab(lab.rename("published_PAM50"), full["call"].rename("genefu_call"))
    conc.to_csv(os.path.join(gdir, "genefu_full_cohort_vs_published_crosstab.tsv"), sep="\t")
    in4 = lab.isin(CLASS_ORDER).to_numpy()
    full_m = metrics(np.array([CLASS_ORDER.index(v) for v in lab[in4]]), genefu_probs(full[in4]))
    exact = float((full["call"].to_numpy()[in4] == lab.to_numpy()[in4]).mean())
    pd.DataFrame([{"scope": "full cohort, 4-class tumours", "n": int(in4.sum()),
                   "exact_agreement_5call": exact, **full_m}]).to_csv(
        os.path.join(gdir, "genefu_full_cohort_concordance.tsv"), sep="\t", index=False)
    print(conc); print(f"\nExact agreement (4-class tumours, genefu 5-call): {exact:.3f} | 4-class macro-F1: {full_m['macro_f1']:.3f}")

per_seed.to_csv(os.path.join(OUT, "per_seed", "per_seed_test_metrics_all_arms.tsv"), sep="\t", index=False)
pred_df.to_csv(os.path.join(OUT, "per_seed", "test_predictions_all_arms.tsv.gz"), sep="\t", index=False)

In [ ]:
# Cell 8 — Summary (mean ± SD over seeds) + paired tests: compact TOP80 against each comparator
ARMS = ["pam50_centroid_v1", "pam50_genefu", "lr_pam50genes", "lr_full3000"] + [f"compact_TOP{k}" for k in PANEL_SIZES]
ARMS = [a for a in ARMS if a in set(per_seed.arm)]
mcols = [c for c in per_seed.columns if c not in ("seed", "arm", "n_called_Normal")]
summary = per_seed.groupby("arm")[mcols].agg(["mean", "std"]).loc[ARMS]
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary.to_csv(os.path.join(OUT, "summary_mean_sd_all_arms.tsv"), sep="\t")
print(summary[[f"{m}_{s}" for m in ("macro_f1", "bacc", "macro_roc_auc", "macro_pr_auc") for s in ("mean", "std")]].round(3))

REF = f"compact_TOP{PRIMARY_K}"
rng = np.random.default_rng(2026)
tests = []
for comp in [a for a in ARMS if a != REF]:
    for s in SEEDS:
        a = pred_df.query("seed == @s and arm == @REF").set_index("SAMPLE_ID")
        b = pred_df.query("seed == @s and arm == @comp").set_index("SAMPLE_ID").loc[a.index]
        y, pa, pb = a.y_true.to_numpy(), a.y_pred.to_numpy(), b.y_pred.to_numpy()
        d0 = f1_score(y, pa, average="macro") - f1_score(y, pb, average="macro")
        bs = []
        for _ in range(N_BOOT):
            i = rng.integers(0, len(y), len(y))
            bs.append(f1_score(y[i], pa[i], average="macro") - f1_score(y[i], pb[i], average="macro"))
        n01, n10 = int(((pa == y) & (pb != y)).sum()), int(((pa != y) & (pb == y)).sum())
        p_mc = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        tests.append({"comparator": comp, "seed": s, "delta_macro_f1": d0,
                      "boot_ci_low": np.percentile(bs, 2.5), "boot_ci_high": np.percentile(bs, 97.5),
                      "compact_right_comp_wrong": n01, "compact_wrong_comp_right": n10, "mcnemar_exact_p": p_mc})
tests = pd.DataFrame(tests)
tests.to_csv(os.path.join(OUT, f"paired_tests_{REF}_vs_comparators.tsv"), sep="\t", index=False)
print(tests.groupby("comparator").agg(mean_delta=("delta_macro_f1", "mean"),
                                      seeds_ci_above_0=("boot_ci_low", lambda x: int((x > 0).sum())),
                                      seeds_mcnemar_p_lt_05=("mcnemar_exact_p", lambda x: int((x < 0.05).sum()))).round(3))

# LumA/LumB confusion pooled over seeds (counts) for each arm
lum = []
for arm in ARMS:
    q = pred_df[pred_df.arm == arm]
    cm = confusion_matrix(q.y_true, q.y_pred, labels=range(4))
    lum.append({"arm": arm, "LumA_to_LumB": cm[2, 3], "LumB_to_LumA": cm[3, 2], "LumB_to_Her2": cm[3, 1],
                "LumA_to_Basal": cm[2, 0], "n_LumA": cm[2].sum(), "n_LumB": cm[3].sum()})
pd.DataFrame(lum).to_csv(os.path.join(OUT, "luminal_confusions_pooled_over_seeds.tsv"), sep="\t", index=False)
print(pd.DataFrame(lum))

In [ ]:
# Cell 9 — Panel stability across seeds (primary 4-class task) + PAM50 overlap
pan = pd.read_csv(os.path.join(OUT, "stability", "compact_TOP80_genes_per_seed.tsv"), sep="\t")
freq = pan.groupby("GENE").agg(seed_count=("seed", "nunique"), mean_rank=("rank", "mean")) \
          .reset_index().sort_values(["seed_count", "mean_rank"], ascending=[False, True])
pam_set = set(pam_names) | {genes[i].upper() for i in pam_idx}
freq["in_PAM50"] = freq.GENE.str.upper().isin(pam_set)
freq.to_csv(os.path.join(OUT, "stability", "gene_inclusion_frequency_4class.tsv"), sep="\t", index=False)

stable = freq[freq.seed_count >= STABLE_MIN]
stable.to_csv(os.path.join(OUT, "stability", f"stable_panel_ge{STABLE_MIN}of{len(SEEDS)}_4class.tsv"), sep="\t", index=False)

stab_sum = {"unique_genes_any_seed": len(freq),
            **{f"genes_in_{k}_of_{len(SEEDS)}": int((freq.seed_count == k).sum()) for k in range(1, len(SEEDS) + 1)},
            "stable_panel_size": len(stable), "stable_in_PAM50": int(stable.in_PAM50.sum()),
            "always_selected": ";".join(sorted(freq[freq.seed_count == len(SEEDS)].GENE)),
            "stable_PAM50_overlap": ";".join(sorted(stable[stable.in_PAM50].GENE))}
pd.Series(stab_sum).to_csv(os.path.join(OUT, "stability", "stability_summary_4class.tsv"), sep="\t", header=False)
for k, v in stab_sum.items(): print(f"{k}: {v}")

In [ ]:
# Cell 10 — Coefficients of the stable panel (seed-123 split, refit TRAIN+VAL)
s = 123
sp = splits[splits.seed == s]
tv = np.array([sid_pos[x] for x in sp[sp.part != "test"].SAMPLE_ID])
gi = np.array([g_up[g.upper()] for g in stable.GENE])
Xs, = fit_prep(X4[tv][:, gi])
m = LogisticRegression(penalty="l2", solver="lbfgs", C=1.0, class_weight="balanced", max_iter=5000,
                       random_state=s).fit(Xs, y4[tv])
coef = pd.DataFrame(m.coef_.T, index=stable.GENE.to_numpy(), columns=CLASS_ORDER)
coef["max_abs"] = coef.abs().max(1)
coef = coef.sort_values("max_abs", ascending=False)
coef.to_csv(os.path.join(OUT, "biology", "stable_panel_LR_coefficients_seed123.tsv"), sep="\t")
top = pd.concat([coef[c].sort_values(ascending=False).head(10).rename("coef").reset_index().assign(subtype=c)
                 for c in CLASS_ORDER]).rename(columns={"index": "GENE"})
top.to_csv(os.path.join(OUT, "biology", "top10_positive_genes_per_subtype.tsv"), sep="\t", index=False)
print(top.groupby("subtype").GENE.apply(lambda x: ", ".join(x.head(5))))

In [ ]:
# Cell 11 — Enrichment of stable non-PAM50 genes (hypergeometric + BH), background = genes that could be selected
#   Gene-set libraries are downloaded from Enrichr as plain text and saved, so the test is fully reproducible.
if RUN_ENRICHR:
    import requests
    from scipy.stats import hypergeom, false_discovery_control

    def get_library(lib):
        fp = os.path.join(OUT, "biology", f"{lib}.gmt")
        if not os.path.exists(fp):
            r = requests.get("https://maayanlab.cloud/Enrichr/geneSetLibrary",
                             params={"mode": "text", "libraryName": lib}, timeout=300)
            r.raise_for_status()
            open(fp, "w").write(r.text)
        sets = {}
        for line in open(fp):
            p = line.rstrip("\n").split("\t")
            if len(p) > 2:
                sets[p[0]] = {g.split(",")[0].strip().upper() for g in p[2:] if g.strip()}
        return sets

    # background: union of the train-only top-V variance genes across seeds
    bg = set()
    for s in SEEDS:
        sp = splits[splits.seed == s]
        tr = np.array([sid_pos[x] for x in sp[sp.part == "train"].SAMPLE_ID])
        v = np.argsort(np.var(SimpleImputer(strategy="median").fit_transform(X4[tr]), 0))[::-1][:V]
        bg |= {g.upper() for g in genes[v]}
    query = set(stable.loc[~stable.in_PAM50, "GENE"].str.upper()) & bg
    print(f"query genes: {len(query)} | background: {len(bg)}")

    res = []
    for lib in ["Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2023"]:
        rows = []
        for term, gs in get_library(lib).items():
            gs = gs & bg
            if not 5 <= len(gs) <= 500:
                continue
            hits = sorted(query & gs)
            if not hits:
                continue
            p = hypergeom.sf(len(hits) - 1, len(bg), len(gs), len(query))
            rows.append({"library": lib, "Term": term, "Overlap": f"{len(hits)}/{len(gs)}",
                         "P-value": p, "Genes": ";".join(hits)})
        t = pd.DataFrame(rows)
        n_tested = sum(5 <= len(gs & bg) <= 500 for gs in get_library(lib).values())
        # BH over every term tested in the library (terms with zero hits get p = 1)
        padj = false_discovery_control(np.r_[t["P-value"].to_numpy(), np.ones(n_tested - len(t))], method="bh")
        res.append(t.assign(**{"Adjusted P-value": padj[:len(t)], "terms_tested": n_tested}))
        print(f"{lib}: {n_tested} terms tested")
    enr = pd.concat(res).sort_values("Adjusted P-value")
    enr.to_csv(os.path.join(OUT, "biology", "enrichment_stable_nonPAM50_bg_filtered.tsv"), sep="\t", index=False)
    print(enr[["library", "Term", "Overlap", "Adjusted P-value", "Genes"]].head(15).to_string(index=False))

In [ ]:
# Cell 12 — Environment record + list of outputs
import sklearn, scipy, platform
open(os.path.join(OUT, "env", "python_versions.txt"), "w").write(
    f"python {platform.python_version()}\nnumpy {np.__version__}\npandas {pd.__version__}\n"
    f"scikit-learn {sklearn.__version__}\nscipy {scipy.__version__}\n")
for root, _, files in sorted(os.walk(OUT)):
    for f in sorted(files):
        print(os.path.relpath(os.path.join(root, f), OUT))